# Bike Rental Forecasting

## Project Overview

Forecasts **daily bike rental count** over a 14-day horizon. Synthetic data spans 2 years with strong weekly cycles, seasonal weather effects, and growth trend.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily bike rental counts, predict the next 14 days. Bike-sharing operators need demand forecasts for fleet rebalancing, maintenance scheduling, and station capacity planning.

## Why This Project Matters

Bike-sharing systems are growing worldwide. Imbalanced stations (too many or too few bikes) frustrate users and reduce revenue. Accurate demand forecasts enable proactive rebalancing and optimal fleet sizing.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily bike rentals
- Strong weekly cycle (weekday commuters, weekend leisure)
- Seasonal pattern (summer peak, winter trough)
- Growth trend (system expansion)
- Weather-driven noise

| Column | Description |
|--------|-------------|
| `date` | Date |
| `rentals` | Daily bike rental count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'rentals'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 3000 + 2 * t  # growth trend
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 500  # weekday commuters
    elif dow == 5: weekly[i] = 800  # Saturday leisure peak
    else: weekly[i] = 400  # Sunday

# Strong summer peak, winter trough
seasonal = 1200 * np.sin(2 * np.pi * (t - 80) / 365)

# Rain-like weather noise (occasionally reduces demand)
weather_shock = np.where(np.random.random(N_POINTS) < 0.15, -800, 0)
noise = np.random.normal(0, 200, N_POINTS)

rentals = base + weekly + seasonal + weather_shock + noise
rentals = np.maximum(rentals, 200).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'rentals': rentals})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,rentals
0,2022-01-01,2701
1,2022-01-02,2144
2,2022-01-03,2393
3,2022-01-04,2757
4,2022-01-05,2523


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('rentals Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("rentals Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

rentals Statistics:
count     730.000000
mean     4136.704110
std       997.385226
min      1225.000000
25%      3373.000000
50%      4131.000000
75%      4920.250000
max      6443.000000
Name: rentals, dtype: float64

CV: 0.241
Skewness: -0.074


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:      798.4 | RMSE:      871.7 | MAPE: 21.07%
Seasonal Naive (7)             | MAE:      404.4 | RMSE:      497.5 | MAPE: 11.86%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared         RMSE  Time Taken
Model                                                                            
KernelRidge                      2890.976382 -221.305876  4349.485830    0.223981
MLPRegressor                     1815.887112 -138.606701  3446.796012    0.974929
LinearSVR                        1353.676770 -103.052059  2975.690712    0.073708
GaussianProcessRegressor          459.090388  -34.237722  1731.674903    0.148838
DecisionTreeRegressor             124.058618   -8.466048   897.525487    0.024452
QuantileRegressor                  70.590498   -4.353115   674.940897    0.425281
DummyRegressor                     67.891512   -4.145501   661.723074    0.015095
NuSVR                              61.565666   -3.658897   629.656981    0.221452
SVR                                60.733164   -3.594859   625.314547    0.041910
ExtraTreeRegressor                 56.130652   -3.240819   600.741090    0.01

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:      309.5 | RMSE:      376.8 | MAPE:  8.79%
FLAML Test (extra_tree)        | MAE:      287.2 | RMSE:      395.0 | MAPE:  8.71%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:      259.5 | RMSE:      343.7 | MAPE:  7.77%
SF AutoETS                     | MAE:      323.0 | RMSE:      374.7 | MAPE:  9.23%
SF SeasonalNaive               | MAE:      326.8 | RMSE:      439.1 | MAPE:  9.92%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                  model        MAE       RMSE      MAPE
           SF AutoARIMA 259.514130 343.688881  7.767150
FLAML Test (extra_tree) 287.189456 394.995687  8.710581
     FLAML (extra_tree) 309.500194 376.772448  8.789443
             SF AutoETS 322.964417 374.669292  9.227345
       SF SeasonalNaive 326.785706 439.120463  9.923747
     Seasonal Naive (7) 404.357143 497.491780 11.855491
     Naive (Last Value) 798.357143 871.705676 21.071239

Top 3:
                  model        MAE       RMSE     MAPE
           SF AutoARIMA 259.514130 343.688881 7.767150
FLAML Test (extra_tree) 287.189456 394.995687 8.710581
     FLAML (extra_tree) 309.500194 376.772448 8.789443


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -11.00, Std: 394.84


## Interpretation and Business Insight

- Bike rentals peak in summer and drop significantly in winter
- Weekday commuter usage provides a stable baseline
- Saturday leisure rides often exceed weekday commuter counts
- Weather shocks (rain days) cause 20-30% demand drops
- System growth adds a steady upward trend

## Limitations

1. Synthetic — real bike data depends heavily on weather, events, infrastructure
2. No weather features (temperature, rain, wind)
3. No station-level data (aggregate only)
4. Daily granularity — hourly matters for rebalancing
5. No event or holiday calendar

## How to Improve This Project

1. Add hourly weather forecasts as features
2. Use station-level data for rebalancing optimization
3. Include event calendar (festivals, sports events)
4. Model hourly demand for real-time dispatch
5. Add competing transport data (transit strikes → bike surge)

## Production Considerations

- Hourly demand forecasting per station
- Integration with weather APIs
- Real-time rebalancing optimization
- Fleet sizing and maintenance scheduling

## Common Mistakes

1. Ignoring weather — temperature and rain explain most variance
2. Not handling winter separately (different demand regime)
3. Using daily totals for hourly rebalancing decisions
4. Treating all stations as identical

## Mini Challenge / Exercises

1. Add a synthetic temperature feature and measure improvement
2. Model weekday vs weekend demand separately
3. Detect rain-day anomalies in the residuals
4. Try monthly aggregation for fleet planning decisions

## Final Summary / Key Takeaways

1. Bike rental demand has strong seasonal and weekly patterns
2. Weather is the dominant short-term driver
3. System growth creates a predictable upward trend
4. Station-level hourly forecasts are needed for operations
5. Combining ML and statistical models gives robust results

In [18]:
import json
metrics = {
    'project': 'Bike Rental Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Bike Rental Forecasting — Complete ===")

Metrics saved.

=== Bike Rental Forecasting — Complete ===
